In [5]:
import pandas as pd

# Read the Excel file with proper header row
df = pd.read_excel('data/SpeciesTreMsPaper_v1_18Mar.xlsx', header=4)  # Row 5 (0-indexed = 4)

# Also read the metadata rows separately
metadata = pd.read_excel('data/SpeciesTreMsPaper_v1_18Mar.xlsx', header=None, nrows=3)

print("=== METADATA ROWS (Categories/Details) ===")
print(metadata)

print("\n=== COLUMN NAMES (Row 5) ===")
print(list(df.columns))

print(f"\nTotal columns: {len(df.columns)}")
print(f"Total data rows: {len(df)}")


=== METADATA ROWS (Categories/Details) ===
               0               1               2               3   \
0  SpeciesDetails  SpeciesDetails  SpeciesDetails  SpeciesDetails   
1            Name            Name             NaN            Taxa   
2  ScientificName     Common Name             NaN           Group   

               4               5               6               7   \
0  SpeciesDetails  SpeciesDetails  SpeciesDetails  SpeciesDetails   
1            Taxa            Taxa          Status          Status   
2           Order          Family             NaN        Priority   

               8                   9   ...             81     82     83  \
0  SpeciesDetails      SpeciesDetails  ...  Microhabitats  EXTRA  EXTRA   
1          Status              Status  ...         SapRun    NaN    NaN   
2      D4/D5Lists  ConservationStatus  ...            NaN    NaN    NaN   

      84     85     86     87     88     89     90  
0  EXTRA  EXTRA  EXTRA  EXTRA  EXTRA  EXTRA  EXTR

In [6]:
# Create a mapping of column → parent category
categories_row1 = metadata.iloc[0].tolist()  # First metadata row
categories_row2 = metadata.iloc[1].tolist()  # Second metadata row
column_names = df.columns.tolist()

# Create a mapping
column_mapping = []
for i, col in enumerate(column_names):
    mapping = {
        'column_name': col,
        'category_row1': categories_row1[i] if i < len(categories_row1) else None,
        'category_row2': categories_row2[i] if i < len(categories_row2) else None
    }
    column_mapping.append(mapping)

# Display as DataFrame for easy viewing
import pandas as pd
mapping_df = pd.DataFrame(column_mapping)
print("\n=== COLUMN MAPPING ===")
print(mapping_df.head(50))  # First 50 columns

# Save to CSV for easier review
mapping_df.to_csv('column_mapping.csv', index=False)
print("\nSaved to column_mapping.csv")



=== COLUMN MAPPING ===
              column_name   category_row1   category_row2
0         Scientific name  SpeciesDetails            Name
1             Common name  SpeciesDetails            Name
2               Full Name  SpeciesDetails             NaN
3                   Class  SpeciesDetails            Taxa
4                   Order  SpeciesDetails            Taxa
5                  Family  SpeciesDetails            Taxa
6              STATUS_ALL  SpeciesDetails          Status
7            PriorityList  SpeciesDetails          Status
8                   D4/D5  SpeciesDetails          Status
9              IUCNStatus  SpeciesDetails          Status
10       ExtinctionTarget  SpeciesDetails          Status
11           Distribution  SpeciesDetails          Status
12   Habitat requirements  SpeciesDetails         Ecology
13       Key requirements  SpeciesDetails         Ecology
14       Functional group  SpeciesDetails         Ecology
15  sensitivity (lichens)  SpeciesDetails       

In [7]:
# Group columns by parent category (row 1)
from collections import defaultdict

category_groups = defaultdict(list)
for i, col in enumerate(column_names):
    parent = categories_row1[i] if i < len(categories_row1) else 'Unknown'
    if pd.notna(parent) and parent != '':
        category_groups[parent].append(col)

print("\n=== COLUMNS BY PARENT CATEGORY ===")
for category, cols in category_groups.items():
    print(f"\n{category} ({len(cols)} columns):")
    print(cols[:10])  # First 10 columns in each group
    if len(cols) > 10:
        print(f"  ... and {len(cols) - 10} more")



=== COLUMNS BY PARENT CATEGORY ===

SpeciesDetails (16 columns):
['Scientific name', 'Common name', 'Full Name', 'Class', 'Order', 'Family', 'STATUS_ALL', 'PriorityList', 'D4/D5', 'IUCNStatus']
  ... and 6 more

HabitatDetails (18 columns):
['Woodland type', 'uses open grown', 'woodland', 'parkland/open grown', 'ToW', 'Orchard', 'Closed / Edge', 'woodland age', 'Conifer', 'Broadleaved']
  ... and 8 more

Microhabitats (47 columns):
['CAVITY PARENT', 'WoodpeckerNesthole', 'TrunkRotHole', 'Trunk-basedRothole', 'HollowTrunk', 'ChimneyTrunk', 'HollowBranch', 'other rot holes', 'DECAYPARENT', 'HeartRot']
  ... and 37 more

EXTRA (9 columns):
['Flowers', 'LightLevels', 'Humidity', 'Layer', 'sheltered', 'Scrub', 'Pantheon', 'OakEcol', 'WoodlandBAP']


In [8]:
# Check what values are used (yes/no/blank?)
print("\n=== VALUE FORMATS ===")
sample_cols = df.columns[10:20]  # Sample middle columns
for col in sample_cols:
    unique_vals = df[col].dropna().unique()
    print(f"\n{col}:")
    print(f"  Unique values: {unique_vals[:5]}")
    print(f"  Non-null count: {df[col].notna().sum()} / {len(df)}")



=== VALUE FORMATS ===

ExtinctionTarget:
  Unique values: <StringArray>
['common', 'yes']
Length: 2, dtype: str
  Non-null count: 37 / 678

Distribution:
  Unique values: <StringArray>
[               'throughout',               'E; Scotland',
 'N; C; SW; Wales; Scotland',        'S; Wales; Scotland',
                  'Scotland']
Length: 5, dtype: str
  Non-null count: 677 / 678

Habitat requirements:
  Unique values: <StringArray>
[                                                                              'Woodland, Urban and Suburban, Farmland, Grassland',
               'Treecreepers need mature trees in which to search the bark's nooks, crannies and fissures for invertebrate food. ',
                                                  'dry bases of sycamore and elm; sheltered; in bases of boulders; shade tolerant',
 'mature broadleaf trees, good supply of fruit, seeds and keys; forages on ground under canopy in winter; need for drinking pools',
                  'Stock Doves enj

In [9]:
# Add this to your notebook:
print("\nTree species values:")
print(df['Tree species'].value_counts().head(20))



Tree species values:
Tree species
oak                68
beech              34
oak, beech         19
birch              17
pine               15
ash                10
elm                 9
beech, ash          6
willow              5
oak, willow         4
apple               4
oak, ash            4
beech, elm          3
ash, beech, oak     3
Pine, Birch         3
oak, pine           3
birch, alder        3
spruce              3
pine, oak           3
elm, oak            2
Name: count, dtype: int64
